# Análise Logística

Integrante: Vitor Hugo Ramos Guerra

**Objetivo:** 

- Análise lead time por etapa (Compra > Aprovação > Postagem > Entrega) e atrasos em relação ao prazo prometido.
- Correlação entre atrasos e nota de avaliação.
- Mapa de calor de desempenho por região.
- Buscar dados externos para enriquecer as analises de dados, como população e renda por UF (IBGE) e dados do e-commerce brasileiro.




In [11]:
import pandas as pd
import numpy as np


## 1. Leitura da base Orders Enriched

**Identificação e Estado**

order_id: O código alfanumérico único de cada pedido.

customer_id: A referência do cliente para essa compra específica.

order_status: O estado atual do envio (na amostra, estão todos "DELIVERED", ou seja, entregues).

Cronograma Logístico (O teu foco principal)

order_purchase_timestamp: A data e hora exata em que o utilizador efetuou a compra.

order_approved_at: A data e hora em que o sistema aprovou o pagamento.

order_delivered_carrier_date: O momento em que a mercadoria foi despachada para a transportadora.

order_delivered_customer_date: O momento exato em que o pacote chegou às mãos do destinatário.

order_estimated_delivery_date: O prazo limite prometido para a entrega da encomenda.

**Valores e Satisfação**

order_price, order_freight, order_total: O valor dos produtos, o custo do frete e o total da transação, respetivamente.

review_score: A nota de avaliação deixada pelo cliente, numa escala de 1 a 5.

**Localização e Agrupamento**

customer_state e customer_city: O estado (UF) e a localidade do cliente, fundamentais para construíres o teu mapa de calor regional.

purchase_year_month: O ano e mês da compra agregados para facilitar análises temporais.

**Métricas Pré-Calculadas**

dt_diff: A margem em dias entre a data prometida e a entrega efetiva.

is_late: O indicador binário que assinala 0 se a encomenda chegou no prazo estipulado, ou 1 se ocorreu um atraso logístico.

In [13]:
orders_enriched = pd.read_csv("C:\\Users\\Vitor H R G\\OneDrive\\Desktop\\FIAP Pós-Tech\\Tech Challenge\\Tech-challenge-POSTECH\\data\\processed\\orders_enriched.csv")

## Converte as datas para o formato datetime
orders_enriched['order_purchase_timestamp'] = pd.to_datetime(orders_enriched['order_purchase_timestamp'])
orders_enriched['order_approved_at'] = pd.to_datetime(orders_enriched['order_approved_at'])
orders_enriched['order_delivered_carrier_date'] = pd.to_datetime(orders_enriched['order_delivered_carrier_date'])
orders_enriched['order_delivered_customer_date'] = pd.to_datetime(orders_enriched['order_delivered_customer_date'])
orders_enriched['order_estimated_delivery_date'] = pd.to_datetime(orders_enriched['order_estimated_delivery_date'])

orders_enriched.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_price,order_freight,order_total,review_score,customer_state,customer_city,purchase_year_month,dt_diff,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,DELIVERED,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,29.99,8.72,38.71,4.0,SP,SAO PAULO,2017-10,-8.0,0.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,DELIVERED,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,118.70,22.76,141.46,4.0,BA,BARREIRAS,2018-07,-6.0,0.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,DELIVERED,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,159.90,19.22,179.12,5.0,GO,VIANOPOLIS,2018-08,-18.0,0.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,DELIVERED,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,45.00,27.20,72.20,5.0,RN,SAO GONCALO DO AMARANTE,2017-11,-13.0,0.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,DELIVERED,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,19.90,8.72,28.62,5.0,SP,SANTO ANDRE,2018-02,-10.0,0.0


## 2. Cálculo da lead time

In [33]:
filtro_cronologico = (
    (orders_enriched["order_delivered_carrier_date"] >= orders_enriched["order_purchase_timestamp"]) &
    (orders_enriched["order_delivered_customer_date"] >= orders_enriched["order_delivered_carrier_date"])
)

orders_enriched_limpo = orders_enriched[filtro_cronologico].copy()

orders_enriched_limpo["lead_time_compras_aprovacao"] = orders_enriched_limpo["order_approved_at"] - orders_enriched_limpo["order_purchase_timestamp"]
orders_enriched_limpo["lead_time_aprovacao_postagem"] = orders_enriched_limpo["order_delivered_carrier_date"] - orders_enriched_limpo["order_approved_at"]
orders_enriched_limpo["lead_time_postagem_entrega"] = orders_enriched_limpo["order_delivered_customer_date"] - orders_enriched_limpo["order_delivered_carrier_date"]
orders_enriched_limpo["atraso_entrega"] = orders_enriched_limpo["order_estimated_delivery_date"] < orders_enriched_limpo["order_delivered_customer_date"]
orders_enriched_limpo.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_price,order_freight,...,review_score,customer_state,customer_city,purchase_year_month,dt_diff,is_late,lead_time_compras_aprovacao,lead_time_aprovacao_postagem,lead_time_postagem_entrega,atraso_entrega
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,DELIVERED,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,29.99,8.72,...,4.0,SP,SAO PAULO,2017-10,-8.0,0.0,0 days 00:10:42,2 days 08:47:45,6 days 01:30:13,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,DELIVERED,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,118.70,22.76,...,4.0,BA,BARREIRAS,2018-07,-6.0,0.0,1 days 06:42:50,0 days 11:06:33,12 days 00:56:45,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,DELIVERED,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,159.90,19.22,...,5.0,GO,VIANOPOLIS,2018-08,-18.0,0.0,0 days 00:16:34,0 days 04:54:37,9 days 04:16:29,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,DELIVERED,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,45.00,27.20,...,5.0,RN,SAO GONCALO DO AMARANTE,2017-11,-13.0,0.0,0 days 00:17:53,3 days 17:54:00,9 days 10:48:43,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,DELIVERED,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,19.90,8.72,...,5.0,SP,SANTO ANDRE,2018-02,-10.0,0.0,0 days 01:01:50,0 days 21:26:05,1 days 22:30:28,False


In [34]:
orders_enriched_limpo.groupby("customer_state")[["lead_time_compras_aprovacao", "lead_time_aprovacao_postagem", "lead_time_postagem_entrega"]].mean()

,lead_time_compras_aprovacao,lead_time_aprovacao_postagem,lead_time_postagem_entrega
customer_state,,,
AC,0 days 15:10:09.987500,2 days 20:19:39.612500,17 days 13:21:36
AL,0 days 11:43:47.909319899,2 days 23:16:37.083123425,21 days 02:02:44.110831234
AM,0 days 09:45:42.710344827,2 days 11:59:36.137931034,23 days 12:28:06.765517241
AP,0 days 15:20:51.701492537,2 days 21:10:09.701492537,23 days 15:55:28.447761194
BA,0 days 11:41:13.491082410,2 days 19:33:35.173124231,16 days 00:50:11.167281672
CE,0 days 11:17:40.197337509,2 days 20:46:08.233359436,17 days 22:30:52.265258216
DF,0 days 10:14:59.134328358,2 days 17:47:54.714973519,9 days 19:26:00.623013962
ES,0 days 10:37:43.562091503,2 days 22:48:06.378079436,12 days 09:56:52.130216189
GO,0 days 12:38:10.006663249,2 days 15:38:54.461814454,12 days 10:47:23.738083034


## 3. Cálculo da correlação


| Valor da Correlação | Tipo de Relação |
| :--- | :--- |
| **`+0.70` a `+1.00`** | Positiva Forte |
| **`+0.30` a `+0.70`** | Positiva Moderada |
| **`-0.30` a `+0.30`** | Fraca ou Nula |
| **`-0.30` a `-0.70`** | Negativa Moderada |
| **`-0.70` a `-1.00`** | Negativa Forte |




In [26]:
correlacao = orders_enriched["atraso_entrega"].corr(orders_enriched["review_score"])
print(f'Correlação entre atraso de entrega e nota de avaliação: {correlacao:.2f}')

Correlação entre atraso de entrega e nota de avaliação: -0.33
